# P22 — DeepSeek-R1: incentivar la capacidad de razonamiento mediante aprendizaje por refuerzo

## 1. Título y paper

**Paper:** *DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning*  
**Autoría:** DeepSeek-AI  
**Año y venue:** 2025 · arXiv:2501.12948 · Nature 645, 633–638 (2025)  
**Nivel:** L5 · **Motor:** `rl_reasoning`  
**Ficha completa:** [`P22_deepseek_r1`](../../papers/foundational/P22_deepseek_r1/README.md)

**Hito:** El razonamiento se incentiva con refuerzo puro, sin trazas humanas anotadas; y es el primer LLM de pesos abiertos publicado tras revisión por pares.

- [arXiv:2501.12948](https://arxiv.org/abs/2501.12948)
- [DOI (Nature 645, 633–638, 2025)](https://doi.org/10.1038/s41586-025-09422-z)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: La cadena de pensamiento dependía de demostraciones humanas caras, y esa supervisión limitaba la capacidad en problemas complejos.
2. Ejecutar una implementación mínima de la propuesta: Recompensar únicamente el RESULTADO verificable y dejar que el comportamiento de razonamiento emerja del refuerzo, para luego transferirlo a modelos menores.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P12
- P13
- P15
- P19


## 4. Intuición

Para enseñar a razonar, la vía cara es que un humano escriba miles de razonamientos ejemplares. La vía de este paper es no escribir ninguno: solo comprobar si la respuesta final es correcta, y dejar que el modelo descubra por sí mismo que verificar antes de responder le renta.


## 5. Concepto mínimo

```text
RLHF (P12):  recompensa = preferencia humana aprendida     → subjetiva, hackeable
Aquí      :  recompensa = ¿la respuesta final es correcta? → objetiva, verificable
```

La señal no dice **cómo** razonar, solo **si acertaste**. El comportamiento de razonamiento —reflexión, verificación, cambio de estrategia— aparece porque aumenta la probabilidad de acertar, no porque nadie lo demostrara.


## 6. Código explicado

El motor entrena una política sobre tres estrategias usando solo la corrección del resultado.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('rl_reasoning', seed=7)['result']
show(r['estrategias'])
print('\nseñal usada:', r['senal_usada'])
for h in r['historia']:
    print(f"it={h['iteracion']:>2} · política={h['politica']} "
          f"· exactitud={h['exactitud_esperada']} · tokens={h['tokens_esperados']}")

## 7. Predicción antes de ejecutar

1. ¿Hacia qué estrategia se desplazará la política?
2. ¿Qué le pasará al coste en tokens mientras sube la exactitud?
3. ¿Funcionaría esto en una tarea donde no se puede comprobar la respuesta?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    h = run_paper_lab('rl_reasoning', seed=semilla)['result']['historia']
    print(f"semilla {semilla:>2} · exactitud {h[0]['exactitud_esperada']} → {h[-1]['exactitud_esperada']} "
          f"· tokens {h[0]['tokens_esperados']} → {h[-1]['tokens_esperados']}")

## 9. Salida interpretable

La exactitud sube y **el coste sube con ella**. Esa es la lectura completa: el razonamiento largo no es gratis, se paga en tokens de inferencia. El cómputo se desplazó del entrenamiento al momento de responder.


## 10. Comentario pedagógico

El límite está en la palabra **verificable**. En matemáticas y código, comprobar la respuesta es barato y objetivo. En redacción, diagnóstico o consejo legal no existe ese verificador, y ahí el método no se traslada sin más. Es la pregunta abierta que este paper deja.


## 11. Error o anti-patrón deliberado

Anti-patrón: leer la traza de razonamiento como si fuera el proceso real del modelo. Es el mismo error que en ReAct, ahora con textos mucho más largos y convincentes.


In [ ]:
print('Una traza larga y segura de si misma NO es una prueba de correccion.')
print('Se optimizo para que la RESPUESTA FINAL sea correcta;')
print('el texto intermedio es un medio, no un certificado auditado.')

## 12. Corrección

Lo que sí se puede afirmar, y cómo se comprueba:


In [ ]:
auditoria = {
    'verificable': 'la respuesta final, contra la solucion conocida',
    'no_verificable_sin_trabajo_extra': 'que cada paso intermedio sea valido',
    'como_comprobarlo': ['ejecutar el codigo generado',
                          'comprobar el resultado numerico por otra via',
                          'muestrear N trazas y ver si concuerdan'],
    'coste': 'cada comprobacion extra es mas computo en inferencia',
}
show(auditoria)

## 13. Desafío guiado

Sube el coste de la estrategia que verifica y comprueba a partir de qué punto deja de compensar.


In [ ]:
estrategias = {'directo': (0.35, 20), 'cadena': (0.60, 90), 'verificacion': (0.82, 240)}
for presupuesto in (50, 120, 300):
    viables = {k: v for k, v in estrategias.items() if v[1] <= presupuesto}
    mejor = max(viables.items(), key=lambda kv: kv[1][0]) if viables else None
    print(f'presupuesto {presupuesto:>3} tokens → mejor viable: {mejor[0] if mejor else "ninguna"} '
          f'(exactitud {mejor[1][0] if mejor else 0})')

## 14. Desafío autónomo

Toma un modelo abierto pequeño y un conjunto de problemas aritméticos con solución conocida. Muestrea k trazas por problema, quédate con las que llegan al resultado correcto y reentrena sobre ellas. Mide exactitud y tokens por respuesta antes y después, y busca el punto donde más cómputo deja de mejorar el resultado.


## 15. Evidencia de aprendizaje

Guarda la curva exactitud/coste, la explicación de por qué la recompensa verificable evita el reward hacking clásico, y el límite de dominios donde no existe verificador.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P22_deepseek_r1/README.md) · evaluación formal: [`assessments/papers/P22_deepseek_r1.md`](../../assessments/papers/P22_deepseek_r1.md)


## 16. Cierre

Aquí termina la ruta ampliada, en 2025. Lo posterior no está consolidado: vive en `frontier/current-topics.yaml` con fecha, y asciende solo cuando cumple los criterios.


## 17. Conexión con el siguiente hito

- frontier/current-topics.yaml

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
